# MIE 690A Week 5-6 Project Setup

Run this notebook before opening a project-track notebook. It checks that your Week-4 data are present, reproduces one simple baseline, and writes a small project configuration file.

## Put these files together

- `P0_Project_Setup.ipynb`
- `w4utils.py`
- `w5_common.py`
- `cavity_data.npz` from Week 4 (or the fixed reference copy supplied in the track ZIP)

You are **not** being asked to rewrite the CFD solver. Your Week-5 task begins from the fixed, quality-checked dataset.

<!-- MIE690A enriched learner edition v2 -->

## How to learn from this notebook

This is a guided computational laboratory, not a script to execute without reading. For every numbered stage:

1. read the physical question and write a prediction;
2. inspect the inputs, outputs, units, and split before running code;
3. run the cell and check assertions/warnings;
4. compare with the stated baseline or physical diagnostic; and
5. write one or two sentences explaining what the result does **and does not** establish.

Use **Restart and Run All** before treating any output as final. Hidden state from out-of-order execution is a reproducibility failure.

### Evidence contract

Keep four kinds of evidence separate:

- **numerical evidence:** residuals, accepted cases, data hashes, grid/time/particle budgets;
- **statistical evidence:** losses, relative errors, variability across seeds/cases;
- **physical evidence:** centerlines, walls, divergence, vortex structure, positivity, moments;
- **computational evidence:** runtime, memory, saved configuration, and machine-readable metrics.

A claim is only as strong as the weakest relevant layer.


## What this setup notebook teaches

The setup is itself a scientific exercise. It verifies that every later project starts from the same numerical object rather than from a file that merely has the right name.

You will learn to distinguish:

- **identity:** the SHA-256 hash tells whether two files have identical bytes;
- **integrity:** shape, finite-value, pressure-gauge, and accepted-case checks tell whether the contents satisfy the data contract;
- **numerical quality:** residual and benchmark columns describe how the CFD labels were produced;
- **project validity:** a declared case-wise split defines what future generalization claim is allowed.

### Expected output

At the end you should have a dataset audit table, a recovered Re = 275 interpolation baseline, a plot with physical diagnostics, and `project_choice.json`. Do not begin a project if any assertion fails.


In [ ]:
# Repository/Colab bootstrap. Run this before the import cell below.
from pathlib import Path
import sys

def _find_course_root(start=Path.cwd()):
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "common" / "w5_common.py").exists():
            return base
    # Colab flat-upload fallback: helper files and data beside the notebook.
    if (start / "w5_common.py").exists():
        return start
    raise FileNotFoundError(
        "Course root not found. Clone the repository, or upload w4utils.py, "
        "w5_common.py, the track-specific helpers, and cavity_data.npz as listed above."
    )

COURSE_ROOT = _find_course_root()
COMMON_DIR = COURSE_ROOT / "common" if (COURSE_ROOT / "common").exists() else COURSE_ROOT
DATASET_PATH = COURSE_ROOT / "data" / "cavity_data.npz"
if not DATASET_PATH.exists():
    DATASET_PATH = COURSE_ROOT / "cavity_data.npz"
sys.path.insert(0, str(COMMON_DIR))
print("Course root:", COURSE_ROOT)
print("Common helpers:", COMMON_DIR)
print("Dataset:", DATASET_PATH)


In [ ]:
from pathlib import Path
import json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import w4utils, w5_common
importlib.reload(w4utils); importlib.reload(w5_common)
print("w4utils:",w4utils.W4_UTILS_VERSION)
print("w5_common:",w5_common.W5_COMMON_VERSION)
print("dataset SHA-256:",w4utils.sha256_file("cavity_data.npz"))

data=w5_common.require_week4_files(str(DATASET_PATH))
print("Available Reynolds numbers:",data["Re"])
print("Stored fields:",[k for k in ("u","v","p","psi","omega") if k in data])
print("Field shape:",data["u"].shape)


## Data schema before the audit

The dataset stores one complete 65×65 field per Reynolds number. Array order is `(case, y, x)`. The primary variables are `u`, `v`, streamfunction `psi`, and vorticity `omega`; pressure is reconstructed and shifted to a zero-mean gauge. The pressure gauge means an arbitrary constant offset is not a prediction error, while a wrong pressure gradient is.

**Prediction prompt:** Which check below would catch a corrupted field full of NaNs? Which would catch a pressure offset? Which would catch convergence to a wrong but finite field? No single check catches all three.


## Dataset audit

The table below confirms which cases were labeled as training and blind test cases in Week 4. Your project may define a new controlled split, but the split must be declared **before** test results are inspected.

In [ ]:
audit=pd.DataFrame({
    "Re":data["Re"],"Week4_split":data["split"],
    "accepted":data.get("accepted",np.ones(len(data["Re"]),dtype=bool)),
    "final_residual":data.get("final_residual",np.full(len(data["Re"]),np.nan))
})
display(audit)
assert np.all(np.isfinite(data["u"])) and np.all(np.isfinite(data["v"]))
assert np.allclose(data["p"].mean(axis=(1,2)),0,atol=1e-8)


## Why interpolation is the recovery baseline

For a smooth one-parameter family, linear field interpolation is cheap, transparent, and often difficult to beat. If `Re_a < Re_* < Re_b`, the baseline is

\[
q(Re_*)=(1-\alpha)q(Re_a)+\alpha q(Re_b),\qquad
\alpha=\frac{Re_*-Re_a}{Re_b-Re_a}.
\]

This operates on complete fields, not randomly selected grid points. It establishes a performance floor for every neural surrogate that uses the same neighboring development cases.

Before running, predict where interpolation error will be largest: the smooth vortex core, the moving-lid corners, or nearly uniform regions. Use the plot to check your reasoning.


## Reproduce a non-neural baseline

This is a recovery check, not your final project. We interpolate the two neighboring Week-4 training fields to predict the withheld case at Re=275 and compute the same numerical and physical metrics used in Week 4.

The wall metrics used in this project exclude the two moving-lid corners, where the lid and side-wall velocity conditions meet discontinuously. Therefore, the CFD truth should have essentially zero wall error.

In [ ]:
train_re=data["Re"][data["split"].astype(str)=="train"]
pred=w5_common.interpolate_case(data,275,train_re)
report=w5_common.evaluate_prediction(data,275,pred)
display(pd.DataFrame([{"method":"field interpolation","Re":275,**report}]))
fig=w5_common.plot_case_evidence(data,275,{"interpolation":pred},"Setup baseline")
plt.show()
truth_idx=int(np.where(data["Re"]==275)[0][0])
truth=(data["u"][truth_idx],data["v"][truth_idx],data["p"][truth_idx])
truth_report=w5_common.evaluate_prediction(data,275,truth)
display(pd.DataFrame([{"method":"CFD truth reference","Re":275,**truth_report}]))


## From topic to falsifiable question

“Use a neural network for cavity flow” is a topic, not a research question. A usable project question names the baseline, controlled modification, test unit, metric, and possible failure.

Example: *When complete Reynolds-number cases are held out, does a wall-weighted coordinate DNN reduce corner-excluded wall RMS error relative to the same uniformly trained network without increasing global velocity error by more than 10%?*

Write the one-sentence question before choosing plots. A figure should answer the question; the question should not be reverse-engineered from the most attractive output.


## Select one project variant

Choose exactly one of the ten variants listed in the guide. You may change this choice only before the Week-5 checkpoint.

In [ ]:
# EDIT THESE THREE LINES.
STUDENT_NAME = "Your Name"
PROJECT_VARIANT = "1A"   # one of 1A,1B,2A,2B,3A,3B,4A,4B,5A,5B
ONE_SENTENCE_QUESTION = "Replace this with your research question."

card={"student":STUDENT_NAME,"variant":PROJECT_VARIANT,
      "research_question":ONE_SENTENCE_QUESTION,
      "dataset":"cavity_data.npz","dataset_cases":data["Re"].tolist()}
w5_common.write_project_card("project_choice.json",card)
print(json.dumps(card,indent=2))


## Submit at the beginning of Week 5

- `project_choice.json`
- one screenshot of the dataset audit
- the interpolation baseline metric table
- your selected project notebook

Do not continue until the baseline runs and you can explain what its error metrics mean.

## Setup concept check

1. Why does a fixed hash not prove that the numerical method is accurate?
2. Why is a random grid-point split easier than a Reynolds-case split?
3. Why is pressure compared after applying a common gauge?
4. If interpolation wins, what scientific conclusion is justified?
5. Which project decision must be frozen before the blind gate?

### Reading

- Ghia, Ghia & Shin (1982) for the cavity centerline benchmark.
- Wilson et al. (2014) and Sandve et al. (2013) for reproducible-computing practice.
- `references/README.md` for the complete annotated route.


## Reproducibility record

Before closing the notebook, record:

- Python and package versions;
- dataset hash and helper versions;
- every physical case in development, validation, and blind sets;
- every seed and candidate value tried;
- the selection rule and when it was frozen;
- output filenames and units; and
- any cell that was skipped, changed, or run with a reduced budget.

Restart the kernel and run all cells in order. If the result changes materially, report the variability instead of selecting the preferred run.

## Troubleshooting without corrupting the experiment

| Symptom | Safe action | Unsafe action |
| --- | --- | --- |
| Missing helper/data file | Re-run the bootstrap and verify paths/hash | Download an unlabeled older copy |
| Training is slow | Use the documented smoke configuration, then label it “smoke” | Quietly reduce epochs/data in the final claim |
| Validation is poor | Inspect scaling, split, and baseline; revise on development data | Open the blind case to choose settings |
| Blind result fails | Report/localize failure and propose a new future experiment | Tune on the blind case while keeping its label “blind” |
| Stochastic result changes | Run multiple declared seeds and report mean/spread | Keep rerunning until one result looks good |
| A neural model loses to interpolation | Verify fairness, then recommend the simpler method | Hide the baseline |

## Final report outline

1. **Question and hypothesis** — one falsifiable sentence.
2. **Data and split** — physical cases, numerical source, and blind unit.
3. **Baseline** — simplest credible comparator using the same allowed information.
4. **Modification** — the one controlled change.
5. **Selection** — validation-only candidates and frozen rule.
6. **Blind numerical result** — aggregate errors and variability.
7. **Physical result** — at least two diagnostics tied to the flow.
8. **Failure or limitation** — where confidence ends.
9. **Cost and reproducibility** — runtime, environment, seeds, saved files.
10. **Conclusion** — helped, hurt, or revealed a tradeoff; no forced positive AI claim.


## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Foundational or supporting notebook; see ARTICLE_FIGURE_MAP.md for its evidence dependency.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../ARTICLE_FIGURE_MAP.md).
